# Random Forest Classification บน Google Colab

Notebook นี้สาธิตกระบวนการสร้างโมเดลแบบครบขั้นตอน

1. นำเข้าข้อมูล  
2. ตรวจสอบและเตรียมข้อมูล  
3. แยก Feature และ Target  
4. แบ่งข้อมูล Train/Test  
5. ทำ Data Preprocessing และ Transform Data  
6. สร้าง Random Forest  
7. ฝึกและประเมินโมเดล  
8. บันทึก Pipeline เป็นไฟล์ `.pkl`  
9. นำโมเดลไปใช้กับ Streamlit


In [ ]:
# ติดตั้งไลบรารีให้ตรงกับเว็บ Streamlit
!pip install -q pandas==2.2.3 numpy==2.3.5 scikit-learn==1.8.0 matplotlib


## ขั้นตอนที่ 1: นำเข้าไลบรารี

In [ ]:
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42


## ขั้นตอนที่ 2: อัปโหลดและอ่านข้อมูล

ให้อัปโหลดไฟล์ `student_performance.csv`


In [ ]:
uploaded_files = files.upload()

csv_filename = next(iter(uploaded_files))
df = pd.read_csv(csv_filename)

print(f"ชื่อไฟล์: {csv_filename}")
print(f"จำนวนแถว: {df.shape[0]:,}")
print(f"จำนวนคอลัมน์: {df.shape[1]:,}")

display(df.head())


## ขั้นตอนที่ 3: ตรวจสอบข้อมูล

In [ ]:
print("ชนิดข้อมูล")
display(df.dtypes.to_frame("dtype"))

print("\nจำนวนค่าที่หาย")
display(df.isna().sum().to_frame("missing_count"))

print("\nจำนวนข้อมูลซ้ำ:", df.duplicated().sum())

# ลบข้อมูลซ้ำ
df = df.drop_duplicates().reset_index(drop=True)

print("\nการกระจายของคลาส")
display(df["passed"].value_counts().to_frame("count"))
display(
    (df["passed"].value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("percent")
)


## ขั้นตอนที่ 4: กำหนด Feature และ Target

In [ ]:
TARGET_COLUMN = "passed"

NUMERIC_FEATURES = [
    "study_hours",
    "attendance_percent",
    "assignment_score",
    "previous_gpa",
]

CATEGORICAL_FEATURES = [
    "internet_access",
    "tutoring",
]

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"ไม่พบคอลัมน์ที่จำเป็น: {missing_columns}"
    )

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print("ขนาด X:", X.shape)
print("ขนาด y:", y.shape)

display(X.head())


## ขั้นตอนที่ 5: แบ่งข้อมูล Train และ Test

ใช้ข้อมูล Train 80% และ Test 20%  
`stratify=y` ช่วยรักษาสัดส่วนของแต่ละคลาส


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {len(X_train):,} แถว")
print(f"Test : {len(X_test):,} แถว")

print("\nสัดส่วนคลาสใน Train")
display(
    (y_train.value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("percent")
)

print("\nสัดส่วนคลาสใน Test")
display(
    (y_test.value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("percent")
)


## ขั้นตอนที่ 6: สร้าง Data Preprocessing

### ข้อมูลตัวเลข

- เติมค่าที่หายด้วย Median
- Transform ด้วย StandardScaler

### ข้อมูลหมวดหมู่

- เติมค่าที่หายด้วยค่าที่พบบ่อยที่สุด
- Transform ด้วย One-Hot Encoding

> Random Forest ไม่จำเป็นต้องทำ Scaling เสมอไป แต่ใส่ไว้เพื่อสาธิตกระบวนการ Transform Data


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor


## ขั้นตอนที่ 7: สร้าง Random Forest และ Pipeline

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", random_forest),
    ]
)

model_pipeline


## ขั้นตอนที่ 8: ฝึกโมเดล

คำสั่ง `fit()` จะเรียนรู้ทั้งค่าที่ใช้ Preprocessing และ Random Forest จากชุด Train เท่านั้น


In [ ]:
model_pipeline.fit(X_train, y_train)

print("ฝึกโมเดลเรียบร้อย")


## ขั้นตอนที่ 9: ตรวจสอบข้อมูลหลัง Transform

In [ ]:
fitted_preprocessor = model_pipeline.named_steps["preprocessor"]

X_train_transformed = fitted_preprocessor.transform(X_train)
transformed_feature_names = fitted_preprocessor.get_feature_names_out()

print("ก่อน Transform:", X_train.shape)
print("หลัง Transform:", X_train_transformed.shape)

print("\nชื่อ Feature หลัง Transform")
for feature_name in transformed_feature_names:
    print(feature_name)


## ขั้นตอนที่ 10: ทำนายและประเมินผล

In [ ]:
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy: {accuracy:.4f} ({accuracy:.2%})")
print(f"ROC-AUC : {roc_auc:.4f}")

print("\nClassification Report")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["ไม่ผ่าน", "ผ่าน"],
        zero_division=0,
    )
)


In [ ]:
confusion = confusion_matrix(y_test, y_pred)

display_confusion = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=["ไม่ผ่าน", "ผ่าน"],
)

display_confusion.plot(values_format="d")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_proba,
)

plt.title("ROC Curve")
plt.show()


## ขั้นตอนที่ 11: Feature Importance

In [ ]:
trained_model = model_pipeline.named_steps["model"]

importance_df = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "importance": trained_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

display(importance_df)

top_features = importance_df.head(12).sort_values(
    "importance",
    ascending=True,
)

plt.figure(figsize=(9, 5))
plt.barh(
    top_features["feature"],
    top_features["importance"],
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()


## ขั้นตอนที่ 12: ทดลองทำนายข้อมูลใหม่

In [ ]:
new_student = pd.DataFrame(
    [{
        "study_hours": 4.5,
        "attendance_percent": 88,
        "assignment_score": 78,
        "previous_gpa": 2.90,
        "internet_access": "Yes",
        "tutoring": "No",
    }]
)

new_prediction = int(
    model_pipeline.predict(new_student)[0]
)

new_probabilities = model_pipeline.predict_proba(
    new_student
)[0]

prediction_label = (
    "มีแนวโน้มสอบผ่าน"
    if new_prediction == 1
    else "มีความเสี่ยงไม่ผ่าน"
)

print("ผลการทำนาย:", prediction_label)
print(f"โอกาสสอบผ่าน: {new_probabilities[1]:.2%}")

display(new_student)


## ขั้นตอนที่ 13: บันทึกโมเดลเป็น `.pkl`

บันทึกทั้ง Preprocessor และ Random Forest รวมกัน เพื่อให้เว็บใช้กระบวนการเดียวกับตอนฝึก


In [ ]:
metrics = {
    "accuracy": float(accuracy),
    "roc_auc": float(roc_auc),
    "confusion_matrix": confusion.tolist(),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
}

model_bundle = {
    "pipeline": model_pipeline,
    "metadata": {
        "project_name": "Student Pass Prediction",
        "model_name": "Random Forest Classifier",
        "target_column": TARGET_COLUMN,
        "feature_columns": FEATURE_COLUMNS,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "class_names": {
            0: "มีความเสี่ยงไม่ผ่าน",
            1: "มีแนวโน้มสอบผ่าน",
        },
        "metrics": metrics,
    },
}

MODEL_FILENAME = "random_forest_model.pkl"

with open(MODEL_FILENAME, "wb") as file:
    pickle.dump(
        model_bundle,
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

with open(
    "model_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(f"บันทึกโมเดลเรียบร้อย: {MODEL_FILENAME}")


## ขั้นตอนที่ 14: ดาวน์โหลดไฟล์โมเดล

In [ ]:
files.download("random_forest_model.pkl")
files.download("model_metrics.json")


## การนำไปใช้กับ Streamlit

นำไฟล์ต่อไปนี้ไว้ในโฟลเดอร์เดียวกัน

```text
app.py
random_forest_model.pkl
requirements.txt
runtime.txt
```

รันด้วยคำสั่ง

```bash
streamlit run app.py
```
